Environment + Requirements

In [1]:
%%bash
set -e

python3 -m pip install -U virtualenv
rm -rf /content/venv
python3 -m virtualenv /content/venv

/content/venv/bin/python --version
/content/venv/bin/pip --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 45.3 MB/s eta 0:00:00
created virtual environment CPython3.12.13.final.0-64-x86_64 in 345ms
  creator CPython3Posix(dest=/content/venv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.0.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
Python 3.12.13
pip 26.0.1 from /content/venv/lib/python3.12/site-packages/pip (python 3.12)


In [2]:
%%bash
set -e

/content/venv/bin/pip install \
  "numpy==1.26.4" \
  "scipy==1.11.4" \
  "scikit-learn==1.3.2" \
  "pandas==2.2.3"

/content/venv/bin/pip install --no-deps "pyarrow==14.0.2"

/content/venv/bin/python -c "import numpy, scipy, sklearn, pandas, pyarrow; print('numpy', numpy.__version__); print('scipy', scipy.__version__); print('sklearn', sklearn.__version__); print('pandas', pandas.__version__); print('pyarrow', pyarrow.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 126.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 59.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 139.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 138.0 MB/s  0:00:00

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 39.1 MB/s  0:00:01
numpy 1.26.4
scipy 1.11.4
sklearn 1.3.2
pandas 2.2.3
pyarrow 14.0.2


In [ ]:
%%bash
set -e

/content/venv/bin/pip install \
  --index-url https://download.pytorch.org/whl/cu121 \
  "torch==2.5.1+cu121" \
  "torchvision==0.20.1+cu121" \
  "torchaudio==2.5.1+cu121"

/content/venv/bin/pip install \
  "huggingface_hub>=0.25,<1.0" \
  "transformers==4.36.2" \
  "tokenizers==0.15.2" \
  "accelerate==0.21.0" \
  "peft==0.7.1" \
  "einops==0.6.1" \
  "sentencepiece==0.1.99" \
  "datasets==2.14.6" \
  "bitsandbytes>=0.46.1" \
  "python-dotenv==1.0.1" \
  "regex==2023.10.3" \
  "requests" \
  "rouge==1.0.1" \
  "safetensors==0.4.5" \
  "Levenshtein==0.25.1" \
  "python-Levenshtein==0.25.1" \
  "Unidecode==1.3.8" \
  "wandb==0.18.3" \
  "matplotlib" \
  "pillow" \
  "tqdm"

/content/venv/bin/python -c "import torch, torchvision, transformers, tokenizers, datasets, bitsandbytes as bnb; print('torch', torch.__version__); print('torchvision', torchvision.__version__); print('transformers', transformers.__version__); print('tokenizers', tokenizers.__version__); print('datasets', datasets.__version__); print('bnb', bnb.__version__); print('cuda', torch.cuda.is_available())"

In [ ]:
%%bash
set -e

cd /content
rm -rf /content/UnLOK-VQA
git clone https://github.com/Vaidehi99/UnLOK-VQA
ls -la /content/UnLOK-VQA | head

**Evaluation**

Download img

In [ ]:
import json, urllib.request
from pathlib import Path

data = json.loads(Path("/content/UnLOK-VQA/data/zsre_mend_eval.json").read_text())
sample = data[:50]

out_dir = Path("/content/UnLOK-VQA/data/coco2017")
(out_dir / "train2017").mkdir(parents=True, exist_ok=True)
(out_dir / "val2017").mkdir(parents=True, exist_ok=True)

base = {
    "train2017": "http://images.cocodataset.org/train2017/",
    "val2017": "http://images.cocodataset.org/val2017/",
}

downloaded = 0
failed = []

for ex in sample:
    image_id = ex["image_id"]
    fn = f"{image_id:012d}.jpg"
    found = False
    for split in ["train2017", "val2017"]:
        out = out_dir / split / fn
        if out.exists():
            found = True
            downloaded += 1
            break
        try:
            urllib.request.urlretrieve(base[split] + fn, out.as_posix())
            found = True
            downloaded += 1
            break
        except:
            pass
    if not found:
        failed.append(image_id)

print("downloaded_or_exists:", downloaded, "/", len(sample))
print("failed:", failed[:20], "..." if len(failed) > 20 else "")

Check images

In [ ]:
import json
from pathlib import Path

data = json.loads(Path("/content/UnLOK-VQA/data/zsre_mend_eval.json").read_text())
for ex in data[:5]:
    iid = ex["image_id"]
    fn = f"{iid:012d}.jpg"
    p1 = Path(f"/content/UnLOK-VQA/data/coco2017/train2017/{fn}")
    p2 = Path(f"/content/UnLOK-VQA/data/coco2017/val2017/{fn}")
    print(ex["id"], iid, p1.exists(), p2.exists())

Minimal eval

In [ ]:
import os
from pathlib import Path

# =========================
# 0) paths / settings
# =========================
repo_dir = Path("/content/UnLOK-VQA")
assert repo_dir.exists(), f"Repo not found: {repo_dir}"
os.chdir(repo_dir)

os.environ["COCO_ROOT"] = "/content/UnLOK-VQA/data/coco2017"

code = r'''
from pathlib import Path
import os
import json
from PIL import Image
from tqdm import tqdm
import torch
import pandas as pd

from transformers import LlavaProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

DATA_PATH = Path("data/zsre_mend_eval.json")
RESULT_PATH = Path("results/minimal_eval_hf_50.jsonl")
MODEL_ID = "llava-hf/llava-1.5-7b-hf"

N = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 1) helper functions
# =========================
COCO_ROOT = Path(os.environ["COCO_ROOT"])

def find_image(image_id: int):
    fn = f"{image_id:012d}.jpg"
    p_train = COCO_ROOT / "train2017" / fn
    p_val = COCO_ROOT / "val2017" / fn
    if p_train.exists():
        return p_train
    if p_val.exists():
        return p_val
    return None

def build_prompt(question: str):
    return f"USER: <image>\n{question}\nASSISTANT:"

def clean_answer(text: str):
    if text is None:
        return None
    text = text.strip()
    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[1].strip()
    return text

def normalize_text(s: str):
    if s is None:
        return ""
    s = s.lower().strip()
    for ch in [".", ",", "!", "?", ":", ";", "\"", "'", "(", ")", "[", "]", "{", "}"]:
        s = s.replace(ch, " ")
    s = " ".join(s.split())
    return s

def get_aliases(target: str):
    t = normalize_text(target)
    aliases = {t}

    # 你可以后面继续扩
    manual = {
        "lab": {"lab", "labrador", "labrador retriever", "black lab", "black labrador"},
        "mountain": {"mountain", "mountains", "mountainous"},
        "mountains": {"mountain", "mountains", "mountainous"},
        "police": {"police", "police station", "cop", "officer"},
        "fire": {"fire", "flame", "burning"},
        "country": {"country", "countryside", "rural area"},
    }

    if t in manual:
        aliases |= manual[t]

    # 简单单复数处理
    if t.endswith("s"):
        aliases.add(t[:-1])
    else:
        aliases.add(t + "s")

    return aliases

def soft_match(target: str, answer: str):
    ans = normalize_text(answer)
    aliases = get_aliases(target)
    return any(alias in ans for alias in aliases)

# dummy image: 用纯白图当“无信息图”
DUMMY_IMAGE = Image.new("RGB", (336, 336), color=(255, 255, 255))

@torch.inference_mode()
def run_with_image(model, processor, question: str, image, max_new_tokens: int = 64):
    prompt = build_prompt(question)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    text = processor.decode(output[0], skip_special_tokens=True)
    return text

# =========================
# 2) load model
# =========================
assert DATA_PATH.exists(), f"Missing {DATA_PATH}"
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
quant_config = BitsAndBytesConfig(load_in_8bit=True)

print("Loading processor...")
processor = LlavaProcessor.from_pretrained(MODEL_ID, use_fast=False)

print("Loading model...")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    quantization_config=quant_config,
    device_map="auto",
)
model.eval()

# =========================
# 3) load data
# =========================
with open(DATA_PATH, "r") as f:
    data = json.load(f)

data = data[:N]

# =========================
# 4) run evaluation
# =========================
rows = []

with open(RESULT_PATH, "w") as fout:
    for ex in tqdm(data):
        q = ex["src"]
        iid = ex["image_id"]
        target = ex.get("pred", "")
        img_path = find_image(iid)

        rec = {
            "id": ex["id"],
            "question": q,
            "target": target,
            "image_id": iid,
            "image_found": img_path is not None,

            "dummy_raw": None,
            "dummy_answer": None,
            "dummy_match_soft": None,

            "true_raw": None,
            "true_answer": None,
            "true_match_soft": None,

            "error": None,
        }

        try:
            # dummy image baseline
            dummy_raw = run_with_image(model, processor, q, DUMMY_IMAGE)
            rec["dummy_raw"] = dummy_raw
            rec["dummy_answer"] = clean_answer(dummy_raw)
            rec["dummy_match_soft"] = soft_match(target, rec["dummy_answer"])

            # true image
            if img_path is not None:
                true_img = Image.open(img_path).convert("RGB")
                true_raw = run_with_image(model, processor, q, true_img)
                rec["true_raw"] = true_raw
                rec["true_answer"] = clean_answer(true_raw)
                rec["true_match_soft"] = soft_match(target, rec["true_answer"])

        except Exception as e:
            rec["error"] = str(e)

        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        rows.append(rec)

print(f"Saved to: {RESULT_PATH.resolve()}")

# =========================
# 5) summary
# =========================
df = pd.DataFrame(rows)

n_total = len(df)
n_img_found = int(df["image_found"].sum())

dummy_rate = df["dummy_match_soft"].astype("boolean").fillna(False).astype(int).mean()
true_rate = df["true_match_soft"].astype("boolean").fillna(False).astype(int).mean()

print("\n===== SUMMARY =====")
print(f"Total samples: {n_total}")
print(f"Images found: {n_img_found}/{n_total}")
print(f"Dummy-image soft hit rate: {dummy_rate:.3f}")
print(f"True-image soft hit rate:  {true_rate:.3f}")

print("\n===== FIRST 5 ROWS =====")
print(df[[
    "id", "target", "image_found",
    "dummy_answer", "dummy_match_soft",
    "true_answer", "true_match_soft",
    "error"
]].head().to_string())

print("\n===== FIRST 3 JSONL LINES =====")
with open(RESULT_PATH, "r") as f:
    for i, line in enumerate(f):
        print(line[:1000])
        if i >= 2:
            break
'''
Path("minimal_eval_unlok.py").write_text(code)
print("saved")

In [ ]:
%%bash
set -e
cd /content/UnLOK-VQA
export COCO_ROOT=/content/UnLOK-VQA/data/coco2017
PYTHONPATH=/content/UnLOK-VQA /content/venv/bin/python minimal_eval_unlok.py